In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

df = pd.read_parquet("/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_st_test_long_corpus_data.parquet")

# remove any rows with empty query or duplicate queries
df = df[df['query'].notna() & (df['query'] != '')]
df = df.drop_duplicates(subset=['query'])

# also remove datapoints with empty context
df = df[df['context'].notna() & (df['context'] != '')]


In [4]:
df.head()

,query,context,type,synthesized,source,metadata,url1,query_tokens,context_tokens,query_num_tokens,context_num_tokens
317305,NASA Earth Day 2021 resources,NASA Disasters Earth Day 2021 Resources | NASA...,search_term-document,True,SDE_general_v3,"{""id"": ""/SDE/nasa_applied_sciences/|https://ap...",https://appliedsciences.nasa.gov/our-impact/ne...,"[nasa, Ġearth, Ġday, Ġ2021, Ġresources]","[nasa, Ġdisasters, Ġearth, Ġday, Ġ2021, Ġresou...",5,1023
1002188,Sierra Nevada watershed characteristics analysis,Description: This publication consists of the ...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2231552984-CEOS_EXTRA"",...",https://cmr.earthdata.nasa.gov/search/concepts...,"[si, erra, Ġnevada, Ġwatershed, Ġcharacteristi...","[description, :, Ġthis, Ġpublication, Ġconsist...",6,1022
854306,NOAA World Data Service for Paleoclimatology,Description: This archived Paleoclimatology St...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2103589364-NOAA_NCEI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...,"[noaa, Ġworld, Ġdata, Ġservice, Ġfor, Ġpale, o...","[description, :, Ġthis, Ġarchived, Ġpale, ocli...",8,1022
1002183,wildland resources center ecosystem reports,Description: This publication consists of the ...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2231552984-CEOS_EXTRA"",...",https://cmr.earthdata.nasa.gov/search/concepts...,"[wild, land, Ġresources, Ġcenter, Ġecosystem, ...","[description, :, Ġthis, Ġpublication, Ġconsist...",6,1022
1166440,Ocean Biogeographical Information System integ...,Description: An international partnership crea...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214612308-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...,"[ocean, Ġbioge, ographical, Ġinformation, Ġsys...","[description, :, Ġan, Ġinternational, Ġpartner...",6,1022


In [5]:
df.shape

(16807, 11)

In [6]:
df["type"].value_counts()

type
search_term-document    15839
title-description         967
question-answer             1
Name: count, dtype: int64

In [7]:
df["source"].value_counts()

source
CMR               13588
SDE_general_v3     2271
SDE_general_v2      931
PDS                  17
Name: count, dtype: int64

In [12]:
dist = df[["type", "source"]].value_counts().sort_index(ascending=False)
dist

type                  source        
title-description     PDS                   3
                      CMR                 964
search_term-document  SDE_general_v3     2270
                      SDE_general_v2      931
                      PDS                  14
                      CMR               12624
question-answer       SDE_general_v3        1
Name: count, dtype: int64

In [23]:
dist.get(('question-answer', 'SDE_general_v2'), 0)  

0

In [24]:
n_title_desc = 750
n_search_doc = 750
n_qa = 0

# subsample for title desc with 1:5 ratio for PDS and CMR
n_title_desc_pds = min(n_title_desc // 6, dist.loc[('title-description', 'PDS')])
n_title_desc_cmr = n_title_desc - n_title_desc_pds

print("For title desc:")
print(f"PDS: {n_title_desc_pds} \nCMR: {n_title_desc_cmr}")

# subsample for search doc with equal ratios
n_search_doc_pds = min(n_search_doc // 4, dist.loc[('search_term-document', 'PDS')])
n_search_doc_cmr = min(n_search_doc // 4, dist.loc[('search_term-document', 'CMR')])
n_search_doc_sde1 = min(n_search_doc // 4, dist.loc[('search_term-document', 'SDE_general_v2')])
n_search_doc_sde2 = n_search_doc - (n_search_doc_pds + n_search_doc_cmr + n_search_doc_sde1)
print("For search doc:")
print(f"PDS: {n_search_doc_pds} \nCMR: {n_search_doc_cmr} \nSDE1: {n_search_doc_sde1} \nSDE2: {n_search_doc_sde2}")

# subsample for qa with equal ratios
n_qa_sde1 = min(n_qa // 2, dist.get(('question-answer', 'SDE_general_v2'), 0))
n_qa_sde2 = min(n_qa - n_qa_sde1, dist.get(('question-answer', 'SDE_general_v3'), 0))
print("For QA:")
print(f"SDE1: {n_qa_sde1} \nSDE2: {n_qa_sde2}")

# taking subsample for title desc
df_title_desc_pds = df[(df["type"] == "title-description") & (df["source"] == "PDS")].sample(n=n_title_desc_pds, random_state=42)
df_title_desc_cmr = df[(df["type"] == "title-description") & (df["source"] == "CMR")].sample(n=n_title_desc_cmr, random_state=42)
df_title_desc = pd.concat([df_title_desc_pds, df_title_desc_cmr], ignore_index=False)


# taking subsample for search doc
df_search_doc_pds = df[(df["type"] == "search_term-document") & (df["source"] == "PDS")].sample(n=n_search_doc_pds, random_state=42)
df_search_doc_cmr = df[(df["type"] == "search_term-document") & (df["source"] == "CMR")].sample(n=n_search_doc_cmr, random_state=42)
df_search_doc_sde1 = df[(df["type"] == "search_term-document") & (df["source"] == "SDE_general_v2")].sample(n=n_search_doc_sde1, random_state=42)
df_search_doc_sde2 = df[(df["type"] == "search_term-document") & (df["source"] == "SDE_general_v3")].sample(n=n_search_doc_sde2, random_state=42)
df_search_doc = pd.concat([df_search_doc_pds, df_search_doc_cmr, df_search_doc_sde1, df_search_doc_sde2], ignore_index=False)

# taking subsample for qa
df_qa_sde1 = df[(df["type"] == "question-answer") & (df["source"] == "SDE_general_v2")].sample(n=n_qa_sde1, random_state=42)
df_qa_sde2 = df[(df["type"] == "question-answer") & (df["source"] == "SDE_general_v3")].sample(n=n_qa_sde2, random_state=42)
df_qa = pd.concat([df_qa_sde1, df_qa_sde2], ignore_index=False)

query_df = pd.concat([df_title_desc, df_search_doc, df_qa], ignore_index=False)

# # get negative corpus: these are datapoints not in df_query
negative_corpus_df = df[~df.index.isin(query_df.index)]




# sampliong negative corpus
# negative_corpus_df = negative_corpus_df.sample(n=50_000, random_state=42)

For title desc:
PDS: 3 
CMR: 747
For search doc:
PDS: 14 
CMR: 187 
SDE1: 187 
SDE2: 362
For QA:
SDE1: 0 
SDE2: 0


In [9]:
# sperating out the meta data

# for query
# - type
# - source
# - synthesized


# for cocrpus
# - source
# - url0


In [25]:
negative_corpus_df.shape, query_df.shape

((15307, 11), (1500, 11))

# converting it to standard jsonal format

In [26]:
import pandas as pd
import json
import os

def create_jsonl_with_negative_corpus(positive_df: pd.DataFrame, negative_df: pd.DataFrame, output_dir: str):
    """
    Generates a dataset for information retrieval, creating corpus, queries, and qrels files.

    This function takes two DataFrames, one with positive query-context pairs and one with
    negative (irrelevant) contexts, and processes them into a structured dataset format.
    It creates three main files:
    1.  `corpus.jsonl`: Contains all unique contexts from both positive and negative dataframes,
        each with a unique ID.
    2.  `queries.jsonl`: Contains all unique queries from the positive dataframe, each with a
        unique ID.
    3.  `qrels/test.tsv`: A tab-separated file mapping query IDs to their relevant
        corpus IDs, based on the positive pairs.

    Args:
        positive_df (pd.DataFrame): DataFrame containing the positive examples.
            Expected columns are: 'query', 'context', 'source', 'url1', 'type', and 'synthesized'.
        negative_df (pd.DataFrame): DataFrame containing negative or irrelevant contexts to be
            added to the corpus. Expected columns include: 'context', 'source', and 'url1'.
        output_dir (str): The path to the directory where the output files will be saved.
            The directory and a 'qrels' subdirectory will be created if they do not exist.
            
    Side Effects:
        - Creates the specified `output_dir` and a `qrels` subdirectory within it.
        - Writes `corpus.jsonl`, `queries.jsonl`, and `qrels/test.tsv` to the disk.
        - Prints status messages to the console during file generation.
    """
    # --- 1. Create Output Directories ---
    qrels_dir = os.path.join(output_dir, 'qrels')
    if not os.path.exists(qrels_dir):
        os.makedirs(qrels_dir)
        print(f"Created directory: {qrels_dir}")

    # --- 2. Process Corpus ---
    # concat the positive and negative corpus dataframes
    total_corpus_df = pd.concat([positive_df, negative_df], ignore_index=False)
    # Get unique contexts to create the corpus
    corpus_df = total_corpus_df[['context', 'source', 'url1']].drop_duplicates(subset=['context']).reset_index(drop=True)
    
    # Create a mapping from context text to a unique corpus ID
    context_to_id = {row['context']: f"c{index}" for index, row in corpus_df.iterrows()}
    
    corpus_filepath = os.path.join(output_dir, 'corpus.jsonl')
    print(f"Generating {corpus_filepath}...")
    with open(corpus_filepath, 'w') as f:
        for index, row in corpus_df.iterrows():
            corpus_id = context_to_id[row['context']]
            corpus_entry = {
                "_id": corpus_id,
                "text": row['context'],
                "metadata": {
                    "source": row['source'],
                    "url": row['url1']
                }
            }
            f.write(json.dumps(corpus_entry) + '\n')
    print(f"Successfully created {corpus_filepath} with {len(corpus_df)} entries.")

    # --- 3. Process Queries ---
    # Get unique queries. Per user, queries will already be unique.
    queries_df = positive_df[['query', 'type', 'source', 'synthesized']].drop_duplicates(subset=['query']).reset_index(drop=True)

    # Create a mapping from query text to a unique query ID
    query_to_id = {row['query']: f"q{index}" for index, row in queries_df.iterrows()}

    queries_filepath = os.path.join(output_dir, 'queries.jsonl')
    print(f"\nGenerating {queries_filepath}...")
    with open(queries_filepath, 'w') as f:
        for index, row in queries_df.iterrows():
            query_id = query_to_id[row['query']]
            query_entry = {
                "_id": query_id,
                "text": row['query'],
                "metadata": {
                    "type": row['type'],
                    "source": row['source'],
                    "synthesized": row['synthesized']
                }
            }
            f.write(json.dumps(query_entry) + '\n')
    print(f"Successfully created {queries_filepath} with {len(queries_df)} entries.")


    # --- 4. Create Qrels (Query-Relevance) File ---
    # lets split the qrels files based on type and source
    _t = positive_df[['type', 'source']].drop_duplicates()
    splits_names = [tuple(row) for row in _t.values]

    for type_name, source_name in splits_names:
        print(f"\nProcessing split: {type_name}~{source_name}")
        qrels_filepath = os.path.join(qrels_dir, f'{type_name}~{source_name}.tsv')
        print(f"\nGenerating {qrels_filepath}...")
        
        # Create a list to hold the relationship data
        qrels_data = []
        # filter positive_df based on type and source
        positive_df_split = positive_df[(positive_df['type'] == type_name) & (positive_df['source'] == source_name)].reset_index(drop=True)
        # Use the original dataframe to preserve all query-context relationships
        for index, row in positive_df_split.iterrows():
            query_id = query_to_id.get(row['query'])
            corpus_id = context_to_id.get(row['context'])
            
            if query_id and corpus_id:
                # The format is: query-id, corpus-id, score (assuming 1)
                qrels_data.append([query_id, corpus_id, 1])

        # Create a DataFrame for qrels and save to TSV without duplicates
        qrels_df = pd.DataFrame(qrels_data, columns=['query-id', 'corpus-id', 'score'])
        qrels_df.drop_duplicates(inplace=True)
        qrels_df.to_csv(qrels_filepath, sep='\t', index=False, header=True)
        
        print(f"Successfully created {qrels_filepath} with {len(qrels_df)} relations.")


In [27]:
output_dir = "/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_long_context_v4/"

create_jsonl_with_negative_corpus(query_df, negative_df=negative_corpus_df, output_dir=output_dir)

Created directory: /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_long_context_v4/qrels
Generating /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_long_context_v4/corpus.jsonl...
Successfully created /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_long_context_v4/corpus.jsonl with 9352 entries.

Generating /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_long_context_v4/queries.jsonl...
Successfully created /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_long_context_v4/queries.jsonl with 1500 entries.

Processing split: title-description~PDS

Generating /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_long_context_v4/qrels/title-description~PDS.tsv..

# testiong the jsonl and qrels files

In [28]:
from datasets import load_dataset

# Load the dataset to verify
corpus = load_dataset(
    "nasa-impact/nasa-sde-IR-benchmark-sample-v4",
    data_files="corpus.jsonl",
    split="train",
)
queries = load_dataset(
    "nasa-impact/nasa-sde-IR-benchmark-sample-v4",
    data_files="queries.jsonl",
    split="train",
)



README.md:   0%|          | 0.00/145 [00:00<?, ?B/s]

corpus.jsonl:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

queries.jsonl:   0%|          | 0.00/269k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [35]:
relevant_docs_data = load_dataset("nasa-impact/nasa-sde-IR-benchmark-sample-v4", data_files="qrels/search_term-document~SDE_general_v3.tsv", split="train")

relevant_docs_data

Dataset({
    features: ['query-id', 'corpus-id', 'score'],
    num_rows: 362
})

In [36]:
relevant_docs_data.to_pandas()

,query-id,corpus-id,score
0,q1138,c1098,1
1,q1139,c1099,1
2,q1140,c1100,1
3,q1141,c1101,1
4,q1142,c1102,1
...,...,...,...
357,q1495,c1432,1
358,q1496,c1433,1
359,q1497,c1434,1
360,q1498,c1435,1


In [13]:
qdf = queries.to_pandas()
cdf = corpus.to_pandas()


In [14]:
qdf

,_id,text,metadata
0,q0,MER Document Collection,"{'type': 'title-description', 'source': 'PDS',..."
1,q1,ROSETTA-ORBITER 67P OSIWAC 2 ESC1-MTP013 EDR V3.0,"{'type': 'title-description', 'source': 'PDS',..."
2,q2,ROSETTA-ORBITER 67P RSI 1/2/3 COMET ESCORT 2 0...,"{'type': 'title-description', 'source': 'PDS',..."
3,q3,RSC-11-9P Collection,"{'type': 'title-description', 'source': 'PDS',..."
4,q4,OMEGA FLIGHT EXPERIMENT DATA RECORDS FROM FIRS...,"{'type': 'title-description', 'source': 'PDS',..."
...,...,...,...
1495,q1495,How does the NASA Earth Science Applied Scienc...,"{'type': 'question-answer', 'source': 'SDE_gen..."
1496,q1496,What is the purpose of the Ozone Hole Watch ac...,"{'type': 'question-answer', 'source': 'SDE_gen..."
1497,q1497,What is the significance of the science goal f...,"{'type': 'question-answer', 'source': 'SDE_gen..."
1498,q1498,What are the primary research interests of Dr....,"{'type': 'question-answer', 'source': 'SDE_gen..."


In [16]:
cdf[cdf["_id"] == "c1425"]["text"].values

array(['With actionable Earth observations, the NASA Earth Science Applied Sciences Program empowers communities across the world to find solutions to the challenges they face every day.'],
      dtype=object)

In [17]:
qdf[qdf["_id"] == "q1495"]["text"].values

array(['How does the NASA Earth Science Applied Sciences Program empower communities with actionable Earth observations?'],
      dtype=object)

In [23]:
x = query_df[['type', 'source']].drop_duplicates()
# x["splits"] = x['type'] + "~" + x['source']
# x["splits"].to_list()

[tuple(row) for row in x.values]


[('title-description', 'PDS'),
 ('title-description', 'CMR'),
 ('search_term-document', 'PDS'),
 ('search_term-document', 'CMR'),
 ('search_term-document', 'SDE_general_v2'),
 ('search_term-document', 'SDE_general_v3'),
 ('question-answer', 'SDE_general_v2'),
 ('question-answer', 'SDE_general_v3')]

In [26]:
type_name, source_name = "search_term-document", "PDS"
query_df[(query_df['type'] == type_name) & (query_df['source'] == source_name)].reset_index(drop=True)

,query,context,type,synthesized,source,metadata,url1
0,Peter Thomas shape models for small solar syst...,Description: The Small Body Shape Models data ...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS4_API/|urn:nasa:pds:ast-sat.th...",
1,Rosetta RPC-LAP raw data,Description: This data set contains EDITEDdata...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS_API_Legacy_All/|464a1db02ff39...",
2,Mars MRS extended mission 7 4160 V1.0,Description: This is a Mars Express Radio Scie...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS_API_Legacy_All/|5b76255f559cf...",
3,EAR-A-3-RDR-TNO-PHOT dataset,Description: This is the document collection f...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS4_API/|urn:nasa:pds:compil.tno...",
4,Mars Express investigation measurements,Description: This is a Mars Express Radio Scie...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS_API_Legacy_All/|c5f29c093ecbc...",
...,...,...,...,...,...,...,...
120,Comet 67P radio science investigation datasets,Description: This is a Rosetta Radio Science d...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS_API_Legacy_All/|392cf70dea718...",
121,LADEE mission context collection,Description: This is the context collection fo...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS4_API/|urn:nasa:pds:ladee_ldex...",
122,Global Gravity measurements from Rosetta,Description: This is a Rosetta Radio Science d...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS_API_Legacy_All/|fcb657aee5f74...",
123,Lunar Prospector mission gravity collection,Description: This collection contains a set of...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS4_API/|urn:nasa:pds:lp_rs_jpl_...",
